In [1]:
import pandas as pd
import geopandas as gpd
import numpy as np
from shapely.geometry import Point
from tqdm import tqdm

# 1. قراءة ملف climate_data_with_elevation.csv
data = pd.read_csv('climate_data_with_elevation.csv')

# 2. استخراج النقاط الفريدة (LAT, LON)
unique_points = data[['LAT', 'LON']].drop_duplicates().reset_index(drop=True)
print(f"عدد النقاط الفريدة: {len(unique_points)}")

# 3. تحويل النقاط إلى GeoDataFrame
geometry = [Point(xy) for xy in zip(unique_points['LON'], unique_points['LAT'])]
gdf_points = gpd.GeoDataFrame(unique_points, geometry=geometry, crs="EPSG:4326")

# 4. قراءة ملف خطوط السواحل من Natural Earth
print("جاري تحميل بيانات السواحل من ne_10m_coastline.shp...")
try:
    coastlines_gdf = gpd.read_file('ne_10m_coastline.shp')
    coastlines_gdf = coastlines_gdf[coastlines_gdf.geometry.type.isin(['LineString', 'MultiLineString'])]
except Exception as e:
    print(f"خطأ أثناء تحميل ne_10m_coastline.shp: {e}")
    print("تأكدي إن الملف 'ne_10m_coastline.shp' موجود في نفس المجلد بتاع الكود")
    exit()

# 5. التأكد من نظام الإحداثيات
coastlines_gdf = coastlines_gdf.set_crs(epsg=4326, allow_override=True)

# 6. التحقق من الأشكال الجغرافية الصالحة
coastlines_gdf = coastlines_gdf[coastlines_gdf.geometry.is_valid]

# 7. التحقق من وجود بيانات السواحل
print(f"عدد خطوط السواحل من Natural Earth: {len(coastlines_gdf)}")
if coastlines_gdf.empty:
    print("لا توجد بيانات سواحل في ملف ne_10m_coastline.shp. تحققي من الملف.")
    exit()

# 8. دالة لتحديد نظام UTM المناسب بناءً على الإحداثيات
def get_utm_zone(lon):
    zone = int((lon + 180) / 6) + 1
    return f"EPSG:326{zone:02d}"

# 9. دالة لحساب المسافة لأقرب ساحل (بالكيلومترات)
def calculate_distance_to_coast(point, coastlines_gdf, lat, lon):
    try:
        if coastlines_gdf.empty:
            return np.nan
        
        # تحديد نظام UTM المناسب
        utm_crs = get_utm_zone(lon)
        
        # تحويل النقطة وخطوط السواحل إلى نظام UTM لحساب المسافة بدقة (بالمتر)
        point_utm = gpd.GeoSeries([point], crs="EPSG:4326").to_crs(utm_crs).iloc[0]
        coastlines_utm = coastlines_gdf.to_crs(utm_crs)
        
        # التحقق من coastlines_utm
        if coastlines_utm.empty:
            print(f"لا توجد بيانات سواحل بعد التحويل لـ UTM للنقطة ({lon}, {lat})")
            return np.nan
        
        # التأكد من الأشكال الجغرافية الصالحة بعد التحويل
        coastlines_utm = coastlines_utm[coastlines_utm.geometry.is_valid]
        if coastlines_utm.empty:
            print(f"لا توجد أشكال جغرافية صالحة للسواحل بعد التحويل لـ UTM للنقطة ({lon}, {lat})")
            return np.nan
        
        # حساب المسافة لأقرب ساحل (بالمتر)
        distances = coastlines_utm.distance(point_utm)
        
        # التحقق من المسافات
        if distances.empty or distances.isna().all():
            print(f"لا توجد مسافات محسوبة للساحل للنقطة ({lon}, {lat})")
            return np.nan
        
        min_distance_meters = distances.min()
        
        # تحويل المسافة من أمتار إلى كيلومترات
        min_distance_km = min_distance_meters / 1000
        
        # طباعة المسافة للتصحيح
        print(f"المسافة لأقرب ساحل للنقطة ({lon}, {lat}): {min_distance_km} كم")
        
        return min_distance_km
    
    except Exception as e:
        print(f"خطأ عند حساب المسافة للساحل للنقطة ({lon}, {lat}): {e}")
        return np.nan

# 10. حساب المسافات لكل النقاط
print(f"حساب المسافات لـ {len(unique_points)} نقاط...")
distances_coast = []

for index, row in tqdm(unique_points.iterrows(), total=len(unique_points), desc="حساب المسافات"):
    # حساب المسافة لأقرب ساحل
    distance_coast = calculate_distance_to_coast(
        gdf_points.loc[index, 'geometry'], 
        coastlines_gdf, 
        row['LAT'], 
        row['LON']
    )
    distances_coast.append(distance_coast)

# 11. تحديث العمود في unique_points
unique_points['Distance_to_Nearest_Coast_km'] = distances_coast

# 12. التحقق من النتايج
print("عينة من النتايج:")
print(unique_points[['LAT', 'LON', 'Distance_to_Nearest_Coast_km']].head(10))

# التحقق من القيم الفاضية
if unique_points['Distance_to_Nearest_Coast_km'].isna().any():
    print(f"تحذير: يوجد قيم فاضية في عمود Distance_to_Nearest_Coast_km")
    print(f"عدد القيم الفاضية: {unique_points['Distance_to_Nearest_Coast_km'].isna().sum()}")

# 13. حفظ النقاط الفريدة مع المسافات في ملف منفصل
unique_points[['LAT', 'LON', 'Distance_to_Nearest_Coast_km']].to_csv("unique_points_with_coast_distance.csv", index=False)
print("تم حفظ النقاط الفريدة مع المسافات في ملف: unique_points_with_coast_distance.csv")

# 14. دمج المسافات مع البيانات الأصلية مع التكرارات
data_with_distance = data.merge(
    unique_points[['LAT', 'LON', 'Distance_to_Nearest_Coast_km']],
    on=['LAT', 'LON'],
    how='left'
)

# 15. حفظ البيانات الأصلية المعدلة مع التكرارات
data_with_distance.to_csv("climate_data_with_elevation.csv", index=False)
print("تم تحديث الملف الأصلي climate_data_with_elevation.csv مع المسافات")

# 16. طباعة أول 5 صفوف للتأكد
print(data_with_distance.head())

عدد النقاط الفريدة: 90
جاري تحميل بيانات السواحل من ne_10m_coastline.shp...
عدد خطوط السواحل من Natural Earth: 4133
حساب المسافات لـ 90 نقاط...


حساب المسافات:   1%|          | 1/90 [00:00<00:21,  4.08it/s]

المسافة لأقرب ساحل للنقطة (25.5, 22.5): 971.7780706469143 كم


حساب المسافات:   2%|▏         | 2/90 [00:00<00:20,  4.34it/s]

المسافة لأقرب ساحل للنقطة (26.5, 22.5): 884.0355323779513 كم


...

حساب المسافات:  99%|█████████▉| 89/90 [00:27<00:00,  2.94it/s]

المسافة لأقرب ساحل للنقطة (30.5, 31.5): 4.4002502930704095 كم


حساب المسافات: 100%|██████████| 90/90 [00:28<00:00,  3.18it/s]

المسافة لأقرب ساحل للنقطة (31.5, 31.5): 4.109542692301111 كم
عينة من النتايج:
    LAT   LON  Distance_to_Nearest_Coast_km
0  22.5  25.5                    971.778071
1  22.5  26.5                    884.035532
2  22.5  27.5                    796.836225
3  22.5  28.5                    709.622648
4  22.5  29.5                    619.174530
5  22.5  30.5                    523.213095
6  22.5  31.5                    423.537678
7  22.5  32.5                    325.213445
8  22.5  33.5                    227.063620
9  22.5  34.5                    131.324225
تم حفظ النقاط الفريدة مع المسافات في ملف: unique_points_with_coast_distance.csv


تم تحديث الملف الأصلي climate_data_with_elevation.csv مع المسافات
    LAT   LON  YEAR  DOY  AIRMASS  ALLSKY_SFC_SW_DWN  ALLSKY_SFC_UVA  \
0  22.5  25.5  2019    1     3.93              14.94            0.81   
1  22.5  26.5  2019    1     4.02              13.70            0.75   
2  22.5  27.5  2019    1     4.14              12.29            0.67   
3  22.5  28.5  2019    1     4.29              11.38            0.62   
4  22.5  29.5  2019    1     4.46              11.21            0.59   

   ALLSKY_SFC_UVB  ALLSKY_SFC_UV_INDEX  CLOUD_AMT  ...  WS2M_MAX  WS2M_MIN  \
0            0.02                 1.12      39.29  ...      5.11      1.45   
1            0.02                 1.04      42.90  ...      5.05      1.61   
2            0.01                 0.91      55.86  ...      3.87      1.63   
3            0.01                 0.85      69.87  ...      3.44      1.33   
4            0.01                 0.80      81.09  ...      3.40      1.26   

   WS50M  WS50M_MAX  WS50M_MIN  